# 1. Synthetic-stage purpose and research questions

The synthetic stage asks whether an externally imposed event effect can be detected; whether predictive direction and the known direct model-imposed lag can be localised; whether the event-response multiplier can be estimated; whether event-aware models improve volatility prediction; and how these results change with parameters, functionals, powered states and schedules.

The frozen convention is

\[
E_i \longrightarrow \omega_{i+1} \longrightarrow X_{i+1}.
\]

Event age 0 affects the first next-state observation; ages 0-12 correspond to direct predictive lags +1 through +13. Known-DGP quantities are retrospective synthetic references only.

# 2. Synthetic experiment map

The map distinguishes deterministic validation, one-seed illustration, repeated-seed robustness, and paired repeated-seed comparison. Notebook 06 supplies reusable infrastructure rather than a standalone scientific result.

In [4]:
from pathlib import Path
import json
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
paths = {
    '08': ROOT / 'notebook08_results' / 'notebook08_numeric_summary.csv',
    '09': ROOT / 'notebook09_results' / 'notebook09_calibration_numeric_summary.csv',
    '10': ROOT / 'notebook10_results' / 'stage4_case_numeric_summary.csv',
    '11metrics': ROOT / 'notebook11_results' / 'run_metrics.csv',
    '11fpr': ROOT / 'notebook11_results' / 'held_out_null_fpr.csv',
    '11thresholds': ROOT / 'notebook11_results' / 'null_max_statistic_thresholds.csv',
    '12': ROOT / '12_log_state_event_decomposition.ipynb',
}
assert all(path.exists() for path in paths.values())
summary08 = pd.read_csv(paths['08'])
summary09 = pd.read_csv(paths['09'])
schedule_numeric = pd.read_csv(paths['10'])
run_metrics = pd.read_csv(paths['11metrics'])
held_out_fpr = pd.read_csv(paths['11fpr'])
null_thresholds = pd.read_csv(paths['11thresholds'])
nb12 = nbformat.read(paths['12'], as_version=4)
assert not any(output.get('output_type') == 'error' for cell in nb12.cells for output in cell.get('outputs', []))
schedule_summary = schedule_numeric.loc[schedule_numeric.metric.isin(['parametric_profile_RMSE','parametric_absolute_coverage_error'])].pivot(index=['case_name','case_label'], columns='metric', values='mean')
schedule_summary['parametric_active_mae_gain'] = schedule_numeric.loc[schedule_numeric.metric.eq('active_sigma_parametric_MAE_gain_pct')].set_index('case_name')['median']
schedule_summary = schedule_summary.reset_index().rename(columns={'parametric_profile_RMSE':'parametric_profile_rmse','parametric_absolute_coverage_error':'parametric_coverage_error'})
assert set(schedule_summary.case_name) == {'dense_periodic','sparse_periodic','sparse_irregular'}
experiment_map = pd.DataFrame([
    ('00', 'Simulator, timing and information structure', 'Deterministic validation', 'final'),
    ('01-05', 'Benchmark detection, recovery and forecasting', 'Canonical one-path workflow', 'final'),
    ('06', 'Reusable workflow infrastructure', 'No standalone scientific result', 'final'),
    ('07', 'Nonlinear functional-form stress test', 'Exploratory one-seed evidence', 'final exploratory'),
    ('08', 'Scalar-parameter and replication robustness', '50-seed repeated evidence', 'final'),
    ('09', 'n=2 and scale-matched comparisons', 'Mixed illustrative and repeated evidence', 'final'),
    ('10', 'Event-frequency and schedule robustness', '50 paired seeds', 'final'),
    ('11', 'Dependence and causality comparison', '50 signal seeds plus 250 held-out null evaluations', 'final'),
    ('12', 'Log-state representation', 'Matched one-seed comparison', 'final exploratory'),
], columns=['notebook', 'main question', 'evidence type', 'status'])
display(experiment_map)

,notebook,main question,evidence type,status
0,00,"Simulator, timing and information structure",Deterministic validation,final
1,01-05,"Benchmark detection, recovery and forecasting",Canonical one-path workflow,final
2,06,Reusable workflow infrastructure,No standalone scientific result,final
3,07,Nonlinear functional-form stress test,Exploratory one-seed evidence,final exploratory
4,08,Scalar-parameter and replication robustness,50-seed repeated evidence,final
5,09,n=2 and scale-matched comparisons,Mixed illustrative and repeated evidence,final
6,10,Event-frequency and schedule robustness,50 paired seeds,final
7,11,Dependence and causality comparison,50 signal seeds plus 250 held-out null evaluat...,final
8,12,Log-state representation,Matched one-seed comparison,final exploratory


# 3. Detection, direction and lag timing

GC provides linear directional predictive evidence when the distributed-lag model and history window are appropriate. A cumulative GC order \(p\) jointly contains lags \(1,\ldots,p\), so a selected order is not an isolated causal lag. Individual conditional-lag tests instead identify the strongest distinguishable direct lag.

Correlation is signed but non-directional association; TDMI is non-negative and non-directional lagged dependence. Correlation and TDMI can look bidirectional because they do not condition on target history in the same way as GC or TE. Transfer entropy is directional conditional information transfer in principle but data-hungry and finite-sample sensitive. Detection is easier than direct model-imposed lag localisation; dense periodic calendars also create wider-lag aliases and reverse predictive artefacts. None of GC, TDMI, or TE proves interventionist causality.

The following rates pool the four non-null synthetic cases for the powered-state change target, giving 200 runs per method. Detection rates should be read alongside the held-out null false-positive rate, because a method that detects frequently may also over-detect under the no-effect system.

In [5]:
null_case = next(case for case in run_metrics.case_name.drop_duplicates() if case == 'No effect (n=1)')
signal_cases = set(run_metrics.case_name.unique()) - {null_case}
methods = ['Lagged correlation', 'Granger causality p=1', 'Granger causality p=13', 'TDMI', 'Transfer entropy']
assert len(signal_cases) == 4 and null_case == 'No effect (n=1)' and set(run_metrics.method) == set(methods)
signal_counts = run_metrics.loc[run_metrics.case_name.isin(signal_cases)].groupby(['case_name','method','transformation']).size()
null_counts = run_metrics.loc[run_metrics.case_name.eq(null_case)].groupby(['case_name','method','transformation']).size()
assert signal_counts.eq(50).all() and null_counts.eq(250).all()
assert null_thresholds.calibration_seed_count.eq(250).all() and held_out_fpr.held_out_runs.eq(250).all()
signal = run_metrics.loc[(run_metrics.transformation == 'changes') & run_metrics.case_name.isin(signal_cases)]
null_held_out = run_metrics.loc[(run_metrics.transformation == 'changes') & run_metrics.case_name.eq(null_case)]
assert len(signal) == 200 * len(methods) and null_held_out.groupby('method').size().eq(250).all()
method_evidence = []
for method in methods:
    d = signal.loc[signal.method.eq(method)]
    detected = d.loc[d.forward_detected.astype(bool)]
    is_gc = method.startswith('Granger causality')
    null = null_held_out.loc[null_held_out.method.eq(method)]
    exported_fpr = held_out_fpr.loc[(held_out_fpr.method.eq(method)) & held_out_fpr.transformation.eq('changes') & held_out_fpr.direction.eq('forward')]
    method_evidence.append({
        'method': method.replace('Granger causality ', 'GC ').replace('Transfer entropy', 'TE'),
        'forward detection rate': d.forward_detected.mean(),
        'forward-only classification rate': d.directional_classification.eq('forward only').mean(),
        'conditional exact +1 localisation rate': np.nan if is_gc else detected.exact_plus_1_recovery.mean(),
        'conditional direct-window localisation rate': np.nan if is_gc else detected.direct_window_recovery.mean(),
        'held-out no-effect false-positive rate': float(exported_fpr.false_positive_rate.iloc[0]) if len(exported_fpr) else null.forward_detected.astype(bool).mean(),
    })
method_evidence = pd.DataFrame(method_evidence)
rate_columns = [column for column in method_evidence if column.endswith('rate')]
display(method_evidence.style.format({column: '{:.1%}' for column in rate_columns}, na_rep='N/A'))
gc_mask = method_evidence['method'].isin(['GC p=1', 'GC p=13'])
assert method_evidence.loc[gc_mask, ['conditional exact +1 localisation rate','conditional direct-window localisation rate']].isna().all().all()
assert all(detected_count == int(signal.loc[signal.method.eq(method), 'forward_detected'].astype(bool).sum()) for method, detected_count in signal.groupby('method').forward_detected.apply(lambda x: x.astype(bool).sum()).items())

,method,forward detection rate,forward-only classification rate,conditional exact +1 localisation rate,conditional direct-window localisation rate,held-out no-effect false-positive rate
0,Lagged correlation,93.0%,21.5%,31.7%,52.2%,5.6%
1,GC p=1,97.5%,57.5%,N/A,N/A,5.2%
2,GC p=13,100.0%,42.0%,N/A,N/A,5.2%
3,TDMI,59.5%,14.5%,25.2%,37.0%,8.8%
4,TE,41.5%,4.5%,31.3%,33.7%,7.2%


For lag-scanning methods, primary localisation is conditional on forward detection: exact +1 localisation means the selected positive-lag peak is +1, and direct-window localisation means it lies in +1,…,+13. GC p=1 instead reports predictive evidence at the pre-specified +1 lag; GC p=13 is a joint direct-window predictive test, not unique-lag localisation. Held-out false-positive rates use 250 no-effect evaluations, with maximum-statistic thresholds frozen from a separate 250 null-calibration simulations.

# 4. Event-profile recovery

The benchmark profile is deliberately smooth and exponential. The parametric profile uses an OLS event-free background; the RF-derived non-parametric profile uses event-age cell means; both event-aware deployed forecasts combine their recovered profile with the common OLS event-free background. Under that studied design, the parametric estimator generally benefits from the correctly specified shape, whereas the non-parametric estimator is more flexible but noisier. Sparse schedules reduce observations per event age and weaken recovery. Good forecasting does not by itself establish accurate event-profile recovery.

No final executed Notebook 08 result supports an event-specific-weight claim, so none is made here.

# 5. Forecasting and calibration

Event-aware forecasts improve active-event-window performance in the synthetic benchmark. Event amplitude and volatility-innovation scale change signal-to-noise and forecast gains; sparse events generally reduce profile precision and improvement. Representative paths are pedagogical, whereas repeated-seed distributions are robustness evidence.

Forecasting is technically closer to a delayed-volatility nowcast: the next spot observation is available before the next volatility state is treated as observed. Calibration requires empirical coverage, coverage error, interval width and interval score together; 100% coverage alone is not necessarily good calibration. Forecast improvement does not guarantee event-profile recovery.

# 6. Robustness findings

Notebook 07 is a focused one-seed stress test, not general nonlinear robustness. Notebook 08 supplies repeated 50-seed parameter evidence: amplitude drives forecast gains, innovation scale governs signal-to-noise, and decay controls persistence. In Notebook 09, \(X_i=\sigma_i^2\) for \(n=2\) and the multiplier acts directly on the powered state; dedicated calibrations are not a pure test of \(n\), while scale-matched comparisons remain calibration-specific.

Notebook 10's final paired evidence uses its regenerated Stage 4 checkpoints and AIC-primary GC summaries. Lower event frequency weakens forward-GC effect size, produces less precise event-profile recovery, and lowers active-window forecast gains. Lag 1 remains the modal strongest direct event lag across all three schedule cases. Dense periodic timing produces stronger wider-window TDMI dependence, consistent with periodic aliases. Sparse periodic versus sparse irregular comparisons should be interpreted cautiously: they help separate regularity from spacing and event-count effects, but do not attribute every sparse-case deterioration to irregularity.

Notebook 11 separates detection, directional classification and conditional direct model-imposed lag localisation: GC p=1 is predictive evidence at the pre-specified +1 lag, GC p=13 is joint, TDMI/correlation detect dependence but struggle with direction, and TE is theoretically directional but weak in finite sparse samples. Notebook 12's conclusion is specific to its tested matched-feature log-state formulation.

In [6]:
def metric_mean(frame, case, metric):
    value = frame.loc[(frame.case_name == case) & (frame.metric == metric), 'mean']
    assert len(value) == 1
    return float(value.iloc[0])

def schedule_value(case, column):
    value = schedule_summary.loc[schedule_summary.case_name.eq(case), column]
    assert len(value) == 1
    return float(value.iloc[0])

schedule_text = '; '.join(
    (
        f"{case.replace('_', ' ')}: "
        f"mean profile RMSE "
        f"{schedule_value(case, 'parametric_profile_rmse'):.5f}, "
        f"median active-window MAE gain "
        f"{schedule_value(case, 'parametric_active_mae_gain'):.1f}%"
    )
    for case in [
        'dense_periodic',
        'sparse_periodic',
        'sparse_irregular',
    ]
)
robustness = pd.DataFrame([
    ('07 nonlinear functionals', 'Frozen workflow remains usable in a focused stress test; numerical result omitted.', 'one-seed exploratory', 'not general nonlinear robustness'),
    ('08 scalar parameters', f"Repeated n=1 benchmark: active MAE gain {metric_mean(summary08,'benchmark','parametric_active_MAE_gain_pct'):.1f}%.", '50 seeds per case', 'chosen parameter grid'),
    (
    '09 powered states',
    (
        f"Dedicated n=2 benchmark mean active-window MAE gain "
        f"{metric_mean(summary09, 'n2_benchmark', 'parametric_active_MAE_gain_pct'):.1f}%."
    ),
    '50 seeds per case',
    (
        'n=2 remains workable under the studied calibrations; this is not a pure effect of changing n because the dedicated n=2 '
        'calibration also differs in scale and other parameters.'
    ),
),
    ('10 schedules', schedule_text + '. Final repeated analysis also reports modal direct lag 1 in every schedule case and stronger dense-periodic wider-window TDMI.', '50 paired seeds', 'three studied calendars'),
    ('11 methods', 'Detection, direction and conditional direct model-imposed lag localisation are different tasks; no universal ranking.', '50 signal seeds plus 250 held-out null evaluations', 'method and representation dependence'),
    ('12 log state', 'The tested matched-feature log-state specification was inferior in this exploratory comparison.', 'one-seed exploratory', 'not all log-state models'),
], columns=['study','main conclusion','evidence strength','key limitation'])
display(robustness)
assert not schedule_summary.empty and all(schedule_value(case, 'parametric_profile_rmse') > 0 for case in schedule_summary.case_name)

,study,main conclusion,evidence strength,key limitation
0,07 nonlinear functionals,Frozen workflow remains usable in a focused st...,one-seed exploratory,not general nonlinear robustness
1,08 scalar parameters,Repeated n=1 benchmark: active MAE gain 22.8%.,50 seeds per case,chosen parameter grid
2,09 powered states,Dedicated n=2 benchmark mean active-window MAE...,50 seeds per case,n=2 remains workable under the studied calibra...
3,10 schedules,"dense periodic: mean profile RMSE 0.00034, med...",50 paired seeds,three studied calendars
4,11 methods,"Detection, direction and conditional direct mo...",50 signal seeds plus 250 held-out null evaluat...,method and representation dependence
5,12 log state,The tested matched-feature log-state specifica...,one-seed exploratory,not all log-state models


# 7. What the synthetic evidence establishes

1. Event-related predictive dependence can be detected under the studied synthetic design.
2. By construction, the first direct model-imposed response occurs at lag +1; lag-scanning methods are evaluated on conditional localisation of that known timing.
3. Distributed persistence makes direct model-imposed lag localisation harder than detection.
4. A smooth parametric profile is advantageous when the true profile is exponential.
5. Event-aware forecasts can improve active-window prediction.
6. Amplitude, noise scale, event spacing and sample size materially affect performance.
7. Periodic schedules can generate aliases.
8. Repeated-seed evidence is needed to distinguish robust findings from path-specific noise.

# 8. What the synthetic evidence does not establish

It does not establish interventionist causality from GC, TDMI or TE alone; universally correct lag localisation; universal superiority of any dependence measure or \(n=2\); or that good forecasts recover the true event mechanism. The tested matched-feature log-state specification is exploratory and does not generalise to all log-volatility models.

Latent synthetic volatility is not observable ATM implied volatility. Empirically, neither the true event-free counterfactual transition nor the true multiplier is observed. One-seed results remain exploratory; repeated results remain conditional on the calibration. The benchmark profile and event timing are deliberately cleaner than real announcement and quote timing.

# 9. Synthetic-to-empirical transition

The synthetic powered-state estimand is

\[
\boxed{X_{t+1}=\omega_{t+1}^{n/2}F_t},\qquad X_t=\sigma_t^n,
\]

with the corresponding decomposition

\[
\boxed{\log X_{t+1}=\frac{n}{2}\log\omega_{t+1}+\log F_t}.
\]

Thus, if a synthetic powered-state coefficient satisfies \(\gamma_a=\frac n2\log\omega(a)\), its inversion is

\[
\boxed{\widehat\omega(a)=\exp\!\left(\frac{2\widehat\gamma_a}{n}\right)}.
\]

For \(n=1\), \(\widehat\omega(a)=\exp(2\widehat\gamma_a)\); for \(n=2\), \(\widehat\omega(a)=\exp(\widehat\gamma_a)\). With \(q=\omega^{n/2}\), the deployed event-aware forecast is \(\widehat X_{i+1}=\widehat q_{i+1}\widehat F_i\), followed by \(\widehat\sigma_{i+1}=\widehat X_{i+1}^{1/n}\).

The synthetic stage establishes what can be validated when the DGP and event weights are known. In observed FX data, the true background and event weights are unobserved, so the empirical stage estimates separate background processes and event responses for realised variance \(\tilde{\sigma}_i^2=\tilde{\omega}_i\tilde F_i\) and implied variance \(\sigma_i^2=\omega_iF_i\), alongside spot log return \(r_i\).

Disagreement between the two estimated event responses may later be investigated as possible information about latent or incompletely reflected event risk. This is an exploratory empirical question, not an established finding, hidden-event discovery, or confirmed trading strategy.